<a href="https://colab.research.google.com/github/tranduchoa518-lang/fake-news-detection/blob/main/notebooks/08_bert_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

from google.colab import drive

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

drive.mount('/content/drive')

print("GPU:", torch.cuda.is_available())

Mounted at /content/drive
GPU: True


In [ ]:
DATA_DIR = "/content/drive/MyDrive/fake-news-detection/data/processed"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].fillna("").astype(str)
    df["label"] = df["label"].astype(int)

print(train_df.head())

                                                text  label
0  THE FACE OF THE DEMOCRAT PARTY Has A Message F...      1
1  INCOMING FRESHMEN Are Put On Notice With Welco...      1
2  REPUBLICANS CALL FOR ANSWERS: Did Wasserman-Sc...      1
3  EXTORTION? HOW IRAN Used Nuke Deal To Force Ob...      1
4  Democrats want strong response to intel report...      0


In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]]
)

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/50493 [00:00<?, ? examples/s]

Map:   0%|          | 0/10820 [00:00<?, ? examples/s]

Map:   0%|          | 0/10821 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
,    logits, labels = eval_pred
,
,    predictions = np.argmax(logits, axis=1)
,
,    return {
,        "accuracy": accuracy_score(labels, predictions),
,        "precision": precision_score(
,            labels, predictions, zero_division=0
,        ),
,        "recall": recall_score(
,            labels, predictions, zero_division=0
,        ),
,        "f1": f1_score(
,            labels, predictions, zero_division=0
,        )
,    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss
100,0.097089
200,0.116757
300,0.066381
400,0.050247
500,0.067474
600,0.103168
700,0.059395
800,0.048153
900,0.064118
1000,0.041570


TrainOutput(global_step=12624, training_loss=0.030304849741769294, metrics={'train_runtime': 1474.0786, 'train_samples_per_second': 68.508, 'train_steps_per_second': 8.564, 'total_flos': 1.328526651829248e+16, 'train_loss': 0.030304849741769294, 'epoch': 2.0})

In [ ]:
# Đánh giá mô hình trên tập kiểm tra
test_results = trainer.evaluate(test_dataset)

print("Kết quả đánh giá BERT:")
print(test_results)

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.013613,0.032489,12624,0.993716,0.995137,0.992634,0.993884


Kết quả đánh giá BERT:
{'eval_loss': 0.032488834112882614, 'eval_accuracy': 0.993715922742815, 'eval_precision': 0.9951368876080692, 'eval_recall': 0.9926338483650736, 'eval_f1': 0.9938837920489296}


In [ ]:
import numpy as np
,from sklearn.metrics import accuracy_score, precision_recall_fscore_support
,
,def compute_metrics(eval_pred):
,    logits, labels = eval_pred
,    predictions = np.argmax(logits, axis=-1)
,
,    precision, recall, f1, _ = precision_recall_fscore_support(
,        labels,
,        predictions,
,        average="binary",
,        zero_division=0
,    )
,
,    accuracy = accuracy_score(labels, predictions)
,
,    return {
,        "accuracy": accuracy,
,        "precision": precision,
,        "recall": recall,
,        "f1": f1
,    }

In [ ]:
evaluation = trainer.evaluate(test_dataset)

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_test = np.array(test_df["label"])

results = {
    "Model": "BERT",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0)
}

print(pd.DataFrame([results]))
print(classification_report(y_test, y_pred))

SAVE_DIR = "/content/drive/MyDrive/fake-news-detection/models/bert"

os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(SAVE_DIR + "/bert_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

print("Đã lưu mô hình BERT!")

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.013613,0.032489,12624,0.993716,0.995137,0.992634,0.993884


  Model  Accuracy  Precision    Recall  F1-score
0  BERT  0.993716   0.995137  0.992634  0.993884
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5255
           1       1.00      0.99      0.99      5566

    accuracy                           0.99     10821
   macro avg       0.99      0.99      0.99     10821
weighted avg       0.99      0.99      0.99     10821



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình BERT!


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/fake-news-detection/results"

os.makedirs(SAVE_DIR, exist_ok=True)

with open(f"{SAVE_DIR}/bert_metrics.json", "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
import os
import json

# Tạo thư mục lưu kết quả
os.makedirs("results", exist_ok=True)
os.makedirs("models/bert", exist_ok=True)

# Lưu mô hình BERT
trainer.save_model("models/bert")
tokenizer.save_pretrained("models/bert")

# Lưu kết quả đánh giá
test_results = trainer.evaluate(test_dataset)

with open("results/bert_metrics.json", "w") as f:
    json.dump(test_results, f, indent=4)

print("Đã lưu mô hình BERT và kết quả đánh giá!")
print(test_results)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.013613,0.032489,12624,0.993716,0.995137,0.992634,0.993884


Đã lưu mô hình BERT và kết quả đánh giá!
{'eval_loss': 0.032488834112882614, 'eval_accuracy': 0.993715922742815, 'eval_precision': 0.9951368876080692, 'eval_recall': 0.9926338483650736, 'eval_f1': 0.9938837920489296}
